In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

DB_PATH  = os.path.join('..', 'data', 'processed', 'zepto.db')
OUT_PATH = os.path.join('..', 'outputs')
os.makedirs(OUT_PATH, exist_ok=True)

conn = sqlite3.connect(DB_PATH)
mart = pd.read_sql('SELECT * FROM analytical_mart', conn)
dlv  = pd.read_sql('SELECT * FROM delivery_features', conn)
conn.close()

print(f'analytical_mart: {mart.shape}')
print(f'delivery_features: {dlv.shape}')

analytical_mart: (10000, 18)
delivery_features: (20000, 4)


In [2]:
# ── Key Stats ─────────────────────────────────────────────────────────────
overall_churn   = mart['churn_flag'].mean() * 100
pct_late        = dlv['is_late'].mean() * 100
mean_delta      = dlv['delivery_delta'].mean()
pct_single      = (mart['total_orders'] == 1).mean() * 100

print('=' * 50)
print(f'  Overall churn rate        : {overall_churn:.1f}%')
print(f'  % deliveries that are late: {pct_late:.1f}%')
print(f'  Mean delivery delta (mins): {mean_delta:.1f}')
print(f'  % single-order customers  : {pct_single:.1f}%')
print('=' * 50)

  Overall churn rate        : 64.7%
  % deliveries that are late: 91.9%
  Mean delivery delta (mins): 31.3
  % single-order customers  : 27.3%


In [3]:
# ── Chart 1: Delivery Delta Distribution ──────────────────────────────────
bins   = [-999, 0, 10, 20, 30, 45, 60, 9999]
labels = ['<0', '0-10', '10-20', '20-30', '30-45', '45-60', '60+']
dlv['delta_bin'] = pd.cut(dlv['delivery_delta'], bins=bins, labels=labels, right=True)

counts = dlv['delta_bin'].value_counts().reindex(labels)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels, counts.values, color='steelblue', edgecolor='white', linewidth=0.6)
ax.bar_label(bars, fmt='%,.0f', padding=3, fontsize=9)
ax.set_title('Delivery Delta Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Delivery Delta Bin (mins over 10-min target)')
ax.set_ylabel('Number of Deliveries')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_PATH, 'delivery_delta_distribution.png'), dpi=150)
plt.close()
print('Saved: delivery_delta_distribution.png')

Saved: delivery_delta_distribution.png


In [4]:
# ── Chart 2: Late Rate by City ─────────────────────────────────────────────
# Join delivery_features with order → customer to get city
conn2 = sqlite3.connect(DB_PATH)
city_dlv = pd.read_sql("""
    SELECT c.city, df.is_late
    FROM delivery_features df
    JOIN [order] o ON df.order_id = o.order_id
    JOIN customer c ON o.customer_id = c.customer_id
""", conn2)
conn2.close()

late_by_city = (city_dlv.groupby('city')['is_late']
                .mean() * 100).sort_values(ascending=False).reset_index()
late_by_city.columns = ['city', 'late_pct']

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(late_by_city['city'], late_by_city['late_pct'],
              color='tomato', edgecolor='white', linewidth=0.6)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=8)
ax.set_title('Late Delivery Rate by City', fontsize=14, fontweight='bold')
ax.set_xlabel('City')
ax.set_ylabel('% Late Orders')
ax.set_ylim(0, late_by_city['late_pct'].max() * 1.15)
plt.xticks(rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_PATH, 'late_rate_by_city.png'), dpi=150)
plt.close()
print('Saved: late_rate_by_city.png')

Saved: late_rate_by_city.png


In [5]:
# ── Chart 3: Churn Rate by Delivery Delta Bin ─────────────────────────────
bins   = [-999, 0, 10, 20, 30, 45, 60, 9999]
labels = ['<0', '0-10', '10-20', '20-30', '30-45', '45-60', '60+']
mart['delta_bin'] = pd.cut(mart['avg_delivery_delta'], bins=bins, labels=labels, right=True)

churn_by_bin = (mart.groupby('delta_bin', observed=True)['churn_flag']
                .mean() * 100).reindex(labels).reset_index()
churn_by_bin.columns = ['delta_bin', 'churn_rate']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(churn_by_bin['delta_bin'], churn_by_bin['churn_rate'],
              color='darkorange', edgecolor='white', linewidth=0.6)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.axhline(mart['churn_flag'].mean() * 100, color='navy', linestyle='--',
           linewidth=1.2, label=f'Overall churn ({mart["churn_flag"].mean()*100:.1f}%)')
ax.set_title('Churn Rate by Avg Delivery Delta Bin', fontsize=14, fontweight='bold')
ax.set_xlabel('Avg Delivery Delta Bin (mins over 10-min target)')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, 100)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_PATH, 'churn_by_delta_bin.png'), dpi=150)
plt.close()
print('Saved: churn_by_delta_bin.png')

Saved: churn_by_delta_bin.png


In [6]:
# ── Chart 4: CLV Tier vs Churn Rate ───────────────────────────────────────
tier_order = ['Low', 'Mid', 'High']
clv_churn = (mart.groupby('clv_tier')['churn_flag']
             .agg(['mean', 'count']).reindex(tier_order).reset_index())
clv_churn.columns = ['clv_tier', 'churn_rate', 'count']
clv_churn['churn_rate'] = clv_churn['churn_rate'] * 100

colors = ['#4878CF', '#6ACC65', '#D65F5F']
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(clv_churn['clv_tier'], clv_churn['churn_rate'],
              color=colors, edgecolor='white', linewidth=0.8, width=0.5)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=11)
# Add count annotation below each bar
for i, (_, row) in enumerate(clv_churn.iterrows()):
    ax.text(i, 2, f'n={int(row["count"]):,}', ha='center', va='bottom',
            fontsize=9, color='white', fontweight='bold')
ax.set_title('Churn Rate by CLV Tier', fontsize=14, fontweight='bold')
ax.set_xlabel('CLV Tier')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_PATH, 'clv_tier_vs_churn.png'), dpi=150)
plt.close()
print('Saved: clv_tier_vs_churn.png')

Saved: clv_tier_vs_churn.png


In [7]:
# ── Chart 5: Correlation Heatmap ──────────────────────────────────────────
num_cols = mart.select_dtypes(include='number').columns.tolist()
corr = mart[num_cols].corr()

# Mask upper triangle
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, linecolor='white',
            annot_kws={'size': 8}, ax=ax)
ax.set_title('Correlation Heatmap — Numeric Features', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_PATH, 'correlation_heatmap.png'), dpi=150)
plt.close()
print('Saved: correlation_heatmap.png')

Saved: correlation_heatmap.png


In [8]:
print('\n=== All 5 charts saved to outputs/ ===')
for f in ['delivery_delta_distribution.png','late_rate_by_city.png',
          'churn_by_delta_bin.png','clv_tier_vs_churn.png','correlation_heatmap.png']:
    path = os.path.join(OUT_PATH, f)
    size = os.path.getsize(path) / 1024
    print(f'  {f:45s} {size:6.1f} KB')


=== All 5 charts saved to outputs/ ===
  delivery_delta_distribution.png                 39.9 KB
  late_rate_by_city.png                           54.0 KB
  churn_by_delta_bin.png                          47.7 KB
  clv_tier_vs_churn.png                           32.7 KB
  correlation_heatmap.png                        137.8 KB
